In [2]:
import os
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    dbname="cinema"
)

cur = conn.cursor()

cur.execute("SELECT current_database();")
print("Connected to:", cur.fetchone()[0])

Connected to: cinema


In [3]:
tables = [
    "movies", "genres", "movie_genres",
    "people", "movie_cast", "movie_crew",
    "ratings", "studios", "movie_studios", "reviews"
]

cur.execute("""
SELECT
    t.table_name,
    COUNT(c.column_name) AS metadata_columns
FROM information_schema.tables t
LEFT JOIN information_schema.columns c
    ON c.table_schema = t.table_schema
   AND c.table_name = t.table_name
   AND c.column_name IN ('created_at', 'modified_at')
WHERE t.table_schema = 'public'
  AND t.table_type = 'BASE TABLE'
GROUP BY t.table_name
ORDER BY t.table_name;
""")

for row in cur.fetchall():
    print(row)

('genres', 0)
('movie_cast', 0)
('movie_crew', 0)
('movie_genres', 0)
('movie_studios', 0)
('movies', 0)
('people', 0)
('ratings', 0)
('reviews', 0)
('studios', 0)


In [4]:
cur.execute("""
CREATE OR REPLACE FUNCTION public.fn_set_modified_at()
RETURNS TRIGGER
LANGUAGE plpgsql
AS $$
BEGIN
    NEW.modified_at = NOW();
    RETURN NEW;
END;
$$;
""")

conn.commit()
print("Shared trigger function created!")

Shared trigger function created!


In [5]:
tables = [
    "movies",
    "genres",
    "movie_genres",
    "people",
    "movie_cast",
    "movie_crew",
    "ratings",
    "studios",
    "movie_studios",
    "reviews"
]

try:
    for table in tables:
        cur.execute(f"""
            ALTER TABLE public.{table}
            ADD COLUMN IF NOT EXISTS
                created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
            ADD COLUMN IF NOT EXISTS
                modified_at TIMESTAMPTZ NOT NULL DEFAULT NOW();
        """)

        cur.execute(f"""
            DROP TRIGGER IF EXISTS
                tr_{table}_set_modified_at
            ON public.{table};

            CREATE TRIGGER tr_{table}_set_modified_at
            BEFORE UPDATE ON public.{table}
            FOR EACH ROW
            EXECUTE FUNCTION public.fn_set_modified_at();
        """)

        print("Configured:", table)

    conn.commit()
    print("All 10 tables configured!")

except Exception:
    conn.rollback()
    raise

Configured: movies
Configured: genres
Configured: movie_genres
Configured: people
Configured: movie_cast
Configured: movie_crew
Configured: ratings
Configured: studios
Configured: movie_studios
Configured: reviews
All 10 tables configured!


In [6]:
cur.execute("""
SELECT
    t.table_name,
    COUNT(c.column_name) AS metadata_columns
FROM information_schema.tables t
LEFT JOIN information_schema.columns c
    ON c.table_schema = t.table_schema
   AND c.table_name = t.table_name
   AND c.column_name IN ('created_at', 'modified_at')
WHERE t.table_schema = 'public'
  AND t.table_type = 'BASE TABLE'
GROUP BY t.table_name
ORDER BY t.table_name;
""")

for row in cur.fetchall():
    print(row)

conn.rollback()

('genres', 2)
('movie_cast', 2)
('movie_crew', 2)
('movie_genres', 2)
('movie_studios', 2)
('movies', 2)
('people', 2)
('ratings', 2)
('reviews', 2)
('studios', 2)


In [8]:
cur.execute("""
CREATE OR REPLACE FUNCTION public.fn_set_modified_at()
RETURNS TRIGGER
LANGUAGE plpgsql
AS $$
BEGIN
    NEW.modified_at = clock_timestamp();
    RETURN NEW;
END;
$$;
""")

conn.commit()

print("Trigger function updated successfully!")

Trigger function updated successfully!


In [9]:
from time import sleep

try:
    cur.execute("""
        INSERT INTO public.genres (genre_name)
        VALUES ('Temporary Trigger Test')
        RETURNING genre_id, created_at, modified_at;
    """)

    before = cur.fetchone()
    print("Before update:", before)

    sleep(1)

    cur.execute("""
        UPDATE public.genres
        SET genre_name = 'Updated Temporary Trigger Test'
        WHERE genre_id = %s
        RETURNING genre_id, created_at, modified_at;
    """, (before[0],))

    after = cur.fetchone()
    print("After update:", after)
    print("modified_at changed:", after[2] > before[2])

finally:
    conn.rollback()

Before update: (2, datetime.datetime(2026, 11, 7, 1, 44, 33, 2955, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 1, 44, 33, 2955, tzinfo=datetime.timezone.utc))
After update: (2, datetime.datetime(2026, 11, 7, 1, 44, 33, 2955, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 1, 44, 34, 6528, tzinfo=datetime.timezone.utc))
modified_at changed: True


In [10]:
cur.execute("""
SELECT
    movie_id AS cleaned_movie_id,
    imdb_id,
    SUBSTRING(imdb_id FROM 3)::BIGINT AS expected_public_id,
    title
FROM schema_clean.vw_movies
WHERE imdb_id ~ '^tt[0-9]+$'
LIMIT 10;
""")

for row in cur.fetchall():
    print(row)

conn.rollback()

(1, 'tt0000009', 9, 'Miss Jerry')
(2, 'tt0000147', 147, 'The Corbett-Fitzsimmons Fight')
(3, 'tt0000502', 502, 'Bohemios')
(4, 'tt0000574', 574, 'The Story of the Kelly Gang')
(5, 'tt0000591', 591, 'The Prodigal Son')
(6, 'tt0000615', 615, 'Robbery Under Arms')
(7, 'tt0000630', 630, 'Hamlet')
(8, 'tt0000675', 675, 'Don Quijote')
(9, 'tt0000679', 679, 'The Fairylogue and Radio-Plays')
(10, 'tt0000838', 838, 'A Cultura do Cacau')


In [11]:
cur.execute("""
SELECT table_name, column_name
FROM information_schema.columns
WHERE table_schema = 'schema_clean'
  AND table_name LIKE 'vw_%'
ORDER BY table_name, ordinal_position;
""")

current_view = None

for view, column in cur.fetchall():
    if view != current_view:
        print(f"\n{view}:")
        current_view = view
    print("  ", column)

conn.rollback()


vw_genres:
   genre_id
   genre_name

vw_movie_cast:
   movie_id
   person_id
   character_name

vw_movie_crew:
   movie_id
   person_id
   job

vw_movie_genres:
   movie_id
   genre_id

vw_movies:
   movie_id
   imdb_id
   title
   release_date
   release_year
   runtime

vw_people:
   person_id
   person_name

vw_ratings:
   movie_id
   rating_value
   rating_source


In [13]:
cur.execute("""
SELECT movie_id, title
FROM schema_clean.vw_movies
ORDER BY movie_id
LIMIT 5;
""")

print(cur.fetchall())
conn.rollback()

[(9, 'Miss Jerry'), (147, 'The Corbett-Fitzsimmons Fight'), (502, 'Bohemios'), (574, 'The Story of the Kelly Gang'), (591, 'The Prodigal Son')]


In [14]:
try:
    cur.execute("""
        INSERT INTO public.movies (
            movie_id,
            title,
            release_date,
            runtime
        )
        SELECT
            movie_id,
            title,
            release_date,
            runtime
        FROM schema_clean.vw_movies
        ORDER BY movie_id
        LIMIT 5
        RETURNING movie_id, title, created_at, modified_at;
    """)

    for row in cur.fetchall():
        print(row)

finally:
    conn.rollback()
    print("Test completed. Changes rolled back.")

(9, 'Miss Jerry', datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc))
(147, 'The Corbett-Fitzsimmons Fight', datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc))
(502, 'Bohemios', datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc))
(574, 'The Story of the Kelly Gang', datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc))
(591, 'The Prodigal Son', datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 7, 2, 1, 35, 20659, tzinfo=datetime.timezone.utc))
Test completed. Changes rolled back.


In [15]:

from psycopg2 import sql

# Final tables in foreign-key dependency order
tables = [
    "movies",
    "genres",
    "people",
    "movie_genres",
    "movie_cast",
    "movie_crew",
    "ratings"
]

# Each view provides data for its corresponding public table
load_queries = [
    ("movies", """
        INSERT INTO public.movies
            (movie_id, title, release_date, runtime)
        SELECT DISTINCT ON (movie_id)
            movie_id, title, release_date, runtime
        FROM schema_clean.vw_movies
        ORDER BY movie_id, title, release_date NULLS LAST
    """),

    ("genres", """
        INSERT INTO public.genres (genre_id, genre_name)
        SELECT DISTINCT genre_id, genre_name
        FROM schema_clean.vw_genres
    """),

    ("people", """
        INSERT INTO public.people (person_id, name)
        SELECT person_id, person_name
        FROM schema_clean.vw_people
    """),

    ("movie_genres", """
        INSERT INTO public.movie_genres (movie_id, genre_id)
        SELECT DISTINCT
            v.movie_id, v.genre_id
        FROM schema_clean.vw_movie_genres v
        JOIN public.movies m
            ON v.movie_id = m.movie_id
        JOIN public.genres g
            ON v.genre_id = g.genre_id
    """),

    ("movie_cast", """
        INSERT INTO public.movie_cast
            (movie_id, person_id, character_name)
        SELECT DISTINCT ON (v.movie_id, v.person_id)
            v.movie_id, v.person_id, v.character_name
        FROM schema_clean.vw_movie_cast v
        JOIN public.movies m
            ON v.movie_id = m.movie_id
        JOIN public.people p
            ON v.person_id = p.person_id
        ORDER BY v.movie_id, v.person_id,
                 v.character_name NULLS LAST
    """),

    ("movie_crew", """
        INSERT INTO public.movie_crew
            (movie_id, person_id, job)
        SELECT DISTINCT
            v.movie_id, v.person_id, v.job
        FROM schema_clean.vw_movie_crew v
        JOIN public.movies m
            ON v.movie_id = m.movie_id
        JOIN public.people p
            ON v.person_id = p.person_id
        WHERE v.job IS NOT NULL
    """),

    ("ratings", """
        INSERT INTO public.ratings
            (movie_id, rating_source, rating_value)
        SELECT DISTINCT ON (v.movie_id)
            v.movie_id, v.rating_source, v.rating_value
        FROM schema_clean.vw_ratings v
        JOIN public.movies m
            ON v.movie_id = m.movie_id
        ORDER BY v.movie_id, v.rating_value DESC
    """)
]

# End any previous read-only transaction
conn.rollback()

try:
    cur.execute("SELECT current_database()")
    db_name = cur.fetchone()[0]

    if db_name != "cinema":
        raise RuntimeError(f"Wrong database: {db_name}")

    # Prevent accidental reloading into populated tables
    for table in tables:
        cur.execute(
            sql.SQL("SELECT EXISTS (SELECT 1 FROM {})").format(
                sql.Identifier("public", table)
            )
        )
        if cur.fetchone()[0]:
            raise RuntimeError(
                f"public.{table} already contains data. "
                "Loading stopped to prevent duplicates."
            )

    # Prevent indefinite waiting on table locks
    cur.execute("SET LOCAL lock_timeout = '5s'")

    print("Starting full data load...", flush=True)

    for table_name, query in load_queries:
        print(f"Loading {table_name}...", flush=True)
        cur.execute(query)
        print(
            f"Inserted {cur.rowcount:,} rows into {table_name}",
            flush=True
        )

    # Save all seven tables together
    conn.commit()
    print("SUCCESS: All seven tables committed!")

except Exception as e:
    conn.rollback()
    print("Load failed. All uncommitted inserts rolled back.")
    print("Error:", e)
    raise


Starting full data load...
Loading movies...
Inserted 759,672 rows into movies
Loading genres...
Inserted 27 rows into genres
Loading people...
Inserted 15,781,953 rows into people
Loading movie_genres...
Inserted 1,046,841 rows into movie_genres
Loading movie_cast...
Inserted 4,092,950 rows into movie_cast
Loading movie_crew...
Inserted 4,580,581 rows into movie_crew
Loading ratings...
Inserted 351,135 rows into ratings
SUCCESS: All seven tables committed!


In [17]:
cur.execute("""
SELECT relname AS table_name,
       n_live_tup AS estimated_rows
FROM pg_stat_user_tables
WHERE schemaname = 'public'
ORDER BY relname;
""")

for row in cur.fetchall():
    print(row)

conn.rollback()

('genres', 27)
('movie_cast', 4092950)
('movie_crew', 4580581)
('movie_genres', 1046841)
('movie_studios', 0)
('movies', 759672)
('people', 15781953)
('ratings', 351135)
('reviews', 0)
('studios', 0)


In [18]:
cur.execute("""
SELECT
    m.movie_id,
    m.title,
    g.genre_name
FROM public.movies m
JOIN public.movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN public.genres g
    ON mg.genre_id = g.genre_id
ORDER BY m.movie_id
LIMIT 10;
""")

for row in cur.fetchall():
    print(row)

conn.rollback()

(9, 'Miss Jerry', 'Romance')
(147, 'The Corbett-Fitzsimmons Fight', 'Documentary')
(147, 'The Corbett-Fitzsimmons Fight', 'News')
(147, 'The Corbett-Fitzsimmons Fight', 'Sport')
(574, 'The Story of the Kelly Gang', 'Action')
(574, 'The Story of the Kelly Gang', 'Adventure')
(574, 'The Story of the Kelly Gang', 'Biography')
(591, 'The Prodigal Son', 'Drama')
(615, 'Robbery Under Arms', 'Drama')
(630, 'Hamlet', 'Drama')
